In [1]:
import tensorflow as tf
import os


In [2]:
# MODEL_PATH = "models/rice_disease_classifier.h5"
# model = tf.keras.models.load_model(MODEL_PATH)

# print(" Model loaded")

# MODEL_PATH = "../models/rice_disease_classifier.h5"
# model = tf.keras.models.load_model(MODEL_PATH)

import os
if os.path.exists("../models/rice_disease_classifier.keras"):
    model = tf.keras.models.load_model("../models/rice_disease_classifier.keras")
    print("Loaded from .keras")
elif os.path.exists("../models/rice_disease_savedmodel"):
    model = tf.keras.models.load_model("../models/rice_disease_savedmodel")
    print("Loaded from SavedModel")
else:
    model = tf.keras.models.load_model("../models/rice_disease_classifier.h5")
    print("Loaded from .h5")

print(" Model loaded")



Loaded from .keras
 Model loaded


In [3]:
# converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Dùng cách này (ổn định hơn)
converter = tf.lite.TFLiteConverter.from_saved_model("../models/rice_disease_savedmodel")
tflite_model = converter.convert()

os.makedirs("models", exist_ok=True)

with open("../models/rice_disease_fp32.tflite", "wb") as f:
    f.write(tflite_model)

print(" Export FP32 TFLite done")


 Export FP32 TFLite done


In [4]:
# converter = tf.lite.TFLiteConverter.from_keras_model(model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]

# tflite_quant_model = converter.convert()

# with open("../models/rice_disease_quant.tflite", "wb") as f:
#     f.write(tflite_quant_model)

# print(" Export Quantized TFLite done")

# Tạo converter với config tương thích
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Chỉ định target_spec để tương thích với runtime cũ
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,  # Chỉ dùng ops chuẩn TFLite
]

#  Đảm bảo input/output là UINT8 (tương thích với tflite_flutter 0.10.1)
def representative_dataset():
    """Cung cấp dataset mẫu cho quantization"""
    import numpy as np
    for _ in range(100):
        yield [np.random.rand(1, 224, 224, 3).astype(np.float32)]

converter.representative_dataset = representative_dataset
converter.inference_input_type = tf.uint8   #  Input là UINT8
converter.inference_output_type = tf.uint8  #  Output là UINT8

# Convert
tflite_quant_model = converter.convert()

# Save
with open("../models/rice_disease_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)

print(" Export Quantized TFLite (UINT8) done")


INFO:tensorflow:Assets written to: C:\Users\qahhn\AppData\Local\Temp\tmprybopq7w\assets


INFO:tensorflow:Assets written to: C:\Users\qahhn\AppData\Local\Temp\tmprybopq7w\assets
C:\Users\qahhn\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\lite\python\convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


 Export Quantized TFLite (UINT8) done


In [5]:
fp32_size = os.path.getsize("../models/rice_disease_fp32.tflite") / (1024*1024)
quant_size = os.path.getsize("../models/rice_disease_quant.tflite") / (1024*1024)

print(f"FP32 size:  {fp32_size:.2f} MB")
print(f"Quant size: {quant_size:.2f} MB")


FP32 size:  9.09 MB
Quant size: 2.74 MB


In [7]:
# import numpy as np
# import cv2

# # Load interpreter
# interpreter = tf.lite.Interpreter(model_path="../models/rice_disease_quant.tflite")
# interpreter.allocate_tensors()

# input_details = interpreter.get_input_details()
# output_details = interpreter.get_output_details()

# # Load 1 ảnh test
# img_path = "../data/split/test/brown_spot/" + os.listdir("../data/split/test/brown_spot")[0]
# img = cv2.imread(img_path)
# img = cv2.resize(img, (224,224))
# img = img / 255.0
# img = np.expand_dims(img, axis=0).astype(np.float32)

# # Run inference
# interpreter.set_tensor(input_details[0]['index'], img)
# interpreter.invoke()

# output = interpreter.get_tensor(output_details[0]['index'])
# print("Prediction:", output)
# print("Predicted class index:", np.argmax(output))

import numpy as np
import cv2

# Load interpreter
interpreter = tf.lite.Interpreter(model_path="../models/rice_disease_quant.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

#  In ra để kiểm tra input/output type
print("Input details:", input_details[0])
print("Output details:", output_details[0])

# Load 1 ảnh test
img_path = "../data/split/test/brown_spot/" + os.listdir("../data/split/test/brown_spot")[0]
img = cv2.imread(img_path)
img = cv2.resize(img, (224, 224))

#  THAY ĐỔI: KHÔNG chia 255, giữ nguyên UINT8
# img = img / 255.0  # XÓA dòng này
img = np.expand_dims(img, axis=0).astype(np.uint8)  # UINT8 thay vì FLOAT32

# Run inference
interpreter.set_tensor(input_details[0]['index'], img)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]['index'])
print("Raw output:", output)

# Convert output từ UINT8 → FLOAT32 để tính probability
output_float = output.astype(np.float32) / 255.0
print("Prediction (probabilities):", output_float)
print("Predicted class index:", np.argmax(output_float))

ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.10 from "C:\Users\qahhn\AppData\Local\Programs\Python\Python310\python.exe"
  * The NumPy version is: "1.23.5"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: The specified module could not be found.


ImportError: numpy._core.multiarray failed to import

In [7]:
labels = [
    "bacterial_blight",
    "barrow_brown_leaf_spot",
    "brown_spot",
    "healthy",
    "leaf_blast",
    "leaf_scald",
    "leaf_smut",
    "neck_blast",
    "rice_hispa",
    "sheath_blight"
    "tungro",
]

with open("../models/labels.txt", "w") as f:
    for label in labels:
        f.write(label + "\n")

print(" labels.txt saved")


 labels.txt saved
